<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/DL_Objdetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To download and unzip the datasets

In [2]:
!kaggle datasets download -d kipshidze/shoplifting-video-dataset


Dataset URL: https://www.kaggle.com/datasets/kipshidze/shoplifting-video-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 726M/726M [00:13<00:00, 56.7MB/s]



In [3]:
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

Dataset path Extraction



In [4]:
!ls /content/local_colab_storage/normal | wc -l

90


In [5]:
!ls /content/local_colab_storage/shoplifting | wc -l

92


In [24]:
import cv2
import os
import numpy as np
import argparse
from imutils import paths
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [7]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)


Frame extraction

In [35]:
def process_video(video_path, max_frames=16, resize_dim=(224, 224)):
    """
    Opens a video, uniformly extracts a fixed number of frames,
    resizes them, and normalizes pixel values.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Handle empty or corrupted videos
    if total_frames <= 0:
        cap.release()
        return None

    # Calculate uniform intervals to pick frames across the whole video duration
    # This ensures a 5-second video and a 20-second video both yield exactly 'max_frames'
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

    video_frames = []

    for frame_idx in frame_indices:
        # Set the reader to the specific frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            break

        # 1. Convert color from BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 2. Resize the frame (e.g., to 224x224)
        frame_resized = cv2.resize(frame, resize_dim)

        # 3. Normalize pixel values by dividing by 255.0 (converts 0-255 integers to 0.0-1.0 floats)
        frame_normalized = frame_resized / 255.0

        video_frames.append(frame_normalized)

    cap.release()

    # If the video didn't have enough readable frames, pad it or skip it
    if len(video_frames) < max_frames:
        return None

    # Convert list of frames into a single NumPy array
    # Output shape: (16, 224, 224, 3)
    return np.array(video_frames, dtype=np.float32)

# --- EXAMPLE USAGE ON ONE FILE ---
# (Replace with your actual unzipped path from !ls)
## sample_path = "./local_colab_storage/"


# 1. Define the video extensions you want to look for
video_extensions = (".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv", ".webm")

# 2. Grab all matching video paths recursively
video_paths = list(paths.list_files(args["dataset"], validExts=video_extensions))
print("video_paths: ", video_paths)

data=[]
labels=[]

#if os.path.exists(video_paths):
for video_path in video_paths:

  label = video_path.split(os.path.sep)[-2]
  labels.append(label)

  processed_tensor = process_video(video_path, max_frames=16, resize_dim=(224, 224))
  if processed_tensor is not None:
    data.append(processed_tensor)

  print("Video Processed Successfully!")
  print(f"Final Tensor Shape: {processed_tensor.shape}") # Expecting (16, 224, 224, 3)
  print(f"Min pixel value: {processed_tensor.min()}, Max pixel value: {processed_tensor.max()}")





video_paths:  ['/content/local_colab_storage/shoplifting/shoplifting-4.mp4', '/content/local_colab_storage/shoplifting/shoplifting-59.mp4', '/content/local_colab_storage/shoplifting/shoplifting-19.mp4', '/content/local_colab_storage/shoplifting/shoplifting-39.mp4', '/content/local_colab_storage/shoplifting/shoplifting-31.mp4', '/content/local_colab_storage/shoplifting/shoplifting-67.mp4', '/content/local_colab_storage/shoplifting/shoplifting-72.mp4', '/content/local_colab_storage/shoplifting/shoplifting-80.mp4', '/content/local_colab_storage/shoplifting/shoplifting-42.mp4', '/content/local_colab_storage/shoplifting/shoplifting-83.mp4', '/content/local_colab_storage/shoplifting/shoplifting-11.mp4', '/content/local_colab_storage/shoplifting/shoplifting-41.mp4', '/content/local_colab_storage/shoplifting/shoplifting-48.mp4', '/content/local_colab_storage/shoplifting/shoplifting-61.mp4', '/content/local_colab_storage/shoplifting/shoplifting-71.mp4', '/content/local_colab_storage/shoplifting

In [30]:
##labels = np.array(labels)

In [36]:
# perform one-hot encoding on the labels
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

In [38]:
# Convert lists to final NumPy arrays
X = np.array(data, dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print(f"Data loading complete!")
print(f"X shape (Videos, Frames, H, W, Channels): {X.shape}")
print(f"y shape (Labels): {y.shape}")


Data loading complete!
X shape (Videos, Frames, H, W, Channels): (182, 16, 224, 224, 3)
y shape (Labels): (182, 2)


Split into Train and Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")